# Exercise 04 — SOLUTION: Text Adversarial Attack

## Background

See student notebook.

In [ ]:
# Install textattack if needed (uncomment):
# !pip install textattack datasets

import warnings, logging, os
warnings.filterwarnings("ignore")
logging.getLogger("textattack").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

test_texts = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film. A complete waste of time and money.",
    "A brilliant masterpiece with stunning performances throughout.",
    "Boring, predictable, and poorly acted from start to finish.",
    "One of the best cinematic experiences I have had in years!",
]

print("=== Baseline Predictions ===")
results = pipe(test_texts)
for text, res in zip(test_texts, results):
    print(f"  [{res['label']:8s} {res['score']:.2f}]  {text[:65]}...")

## TextFooler Attack — Solution

In [ ]:
# SOLUTION

# Patch transformers.utils for textattack compatibility (newer transformers removed is_tf_available)
import transformers.utils as _tu
if not hasattr(_tu, 'is_tf_available'):
    _tu.is_tf_available = lambda: False
if not hasattr(_tu, 'is_torch_available'):
    _tu.is_torch_available = lambda: True

from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.attack_recipes import TextFoolerJin2019
from textattack import Attacker, AttackArgs
from textattack.datasets import Dataset as TADataset

model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
attack = TextFoolerJin2019.build(model_wrapper)

labels_int = [1 if r['label'] == 'POSITIVE' else 0 for r in results]
dataset = TADataset(list(zip(test_texts, labels_int)))

attack_args = AttackArgs(num_examples=5, disable_stdout=True, silent=True)
attacker = Attacker(attack, dataset, attack_args)
attack_results = attacker.attack_dataset()

LABEL_MAP = {0: 'NEGATIVE', 1: 'POSITIVE'}
rows = []
for res in attack_results:
    orig = res.original_result.attacked_text.text
    perturbed = res.perturbed_result.attacked_text.text
    orig_label = LABEL_MAP.get(res.original_result.output, str(res.original_result.output))
    new_label  = LABEL_MAP.get(res.perturbed_result.output, str(res.perturbed_result.output))
    outcome = type(res).__name__.replace('AttackResult','')
    orig_words = orig.split(); perturbed_words = perturbed.split()
    changed = sum(1 for a,b in zip(orig_words, perturbed_words) if a != b)
    pct_changed = 100 * changed / max(len(orig_words), 1)
    rows.append({'Original': orig[:70], 'Adversarial': perturbed[:70],
                 'Orig label': orig_label, 'New label': new_label,
                 'Outcome': outcome, '% words changed': f'{pct_changed:.0f}%'})

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 72)
print(df.to_string(index=False))

## Discussion

In [ ]:
print("""
=== Discussion ===

1. Are the adversarial examples still readable?
   TextFooler substitutes synonyms, so the text usually remains grammatical and
   semantically similar to a human reader — yet flips the model's prediction.

2. Why does synonym substitution work?
   Neural models are sensitive to specific token representations. A synonym that is
   semantically equivalent to a human may have a very different embedding, causing
   the model to cross a decision boundary.

3. How could you defend against this?
   - Adversarial training with synonym substitutions
   - Certified defenses based on interval bound propagation
   - Ensemble models trained on paraphrases
""")
